# Generador de Matrículas Españolas con Fechas ITV

Este notebook genera matrículas españolas en formato **NNNNLLL** (4 dígitos + 3 consonantes sin espacios) con fechas de caducidad de ITV.

## Lógica de generación:

- **Letras < M**: Coches antiguos
  - 5% con ITV caducada (fecha pasada)
  - 95% con ITV vigente hasta máximo 12/05/2026
  
- **Letras ≥ M**: Coches nuevos (Serie M+)
  - ITV calculada como 4 años después de la fecha de matriculación
  - Matriculaciones distribuidas entre 01/01/2022 y 05/12/2025

In [ ]:
# Import Required Libraries
import csv
import random
from datetime import date, timedelta
import argparse

## Configuración de Variables

Definimos las constantes y parámetros de configuración:

In [ ]:
# Configuración de Variables
ALLOWED_LETTERS = list("BCDFGHJKLMNPRSTVWXYZ")  # sin vocales, sin Ñ, sin Q
ALLOWED_FIRST = [L for L in ALLOWED_LETTERS if L <= 'N']

TARGET_PLATE = "5500NHZ"     # límite inclusive
TODAY = date(2025, 12, 5)
MAX_ITV_DATE = date(2026, 5, 12)  # máximo para coches antiguos
SERIE_M_START = date(2022, 1, 1)  # inicio serie M
SERIE_M_END = TODAY                # última matrícula conocida

print(f"Configuración:")
print(f"  - Matrícula objetivo: {TARGET_PLATE}")
print(f"  - Fecha actual: {TODAY}")
print(f"  - ITV máxima coches antiguos: {MAX_ITV_DATE}")
print(f"  - Serie M+ desde: {SERIE_M_START} hasta {SERIE_M_END}")

## Funciones Auxiliares

Funciones para generar matrículas y fechas:

In [ ]:
def plate_key(num, l1, l2, l3):
    """Genera clave de matrícula en formato NNNNLLL"""
    return f"{num:04d}{l1}{l2}{l3}"

def should_be_expired(prob=0.05):
    """Determina si una matrícula debe estar caducada (5% probabilidad)"""
    return random.random() < prob

def random_past_date(max_years=5):
    """Genera fecha pasada aleatoria"""
    days = random.randint(1, max_years * 365)
    return TODAY - timedelta(days=days)

def random_future_date():
    """Genera fecha futura aleatoria hasta MAX_ITV_DATE"""
    days = random.randint(0, (MAX_ITV_DATE - TODAY).days)
    return TODAY + timedelta(days=days)

def calculate_serie_m_distribution(total_matriculas):
    """Calcula distribución diaria para la serie M+"""
    total_days = (SERIE_M_END - SERIE_M_START).days + 1
    matriculas_por_dia = max(1, total_matriculas // total_days)
    return matriculas_por_dia, total_days

def is_m_plus(first_letter):
    return first_letter >= 'M'

def iter_m_series_until_target(target_plate):
    """Genera todas las matrículas de la serie M+ hasta alcanzar la matrícula objetivo (inclusive)."""
    for l1 in ALLOWED_LETTERS:
        if l1 < 'M':
            continue
        for l2 in ALLOWED_LETTERS:
            for l3 in ALLOWED_LETTERS:
                for num in range(0, 10000):
                    plate = plate_key(num, l1, l2, l3)
                    yield plate, l1
                    if plate == target_plate:
                        return

def count_m_series_until_target(target_plate):
    """Cuenta cuántas matrículas M+ hay hasta la matrícula objetivo (inclusive)."""
    return sum(1 for _ in iter_m_series_until_target(target_plate))

def iter_all_plates_until_target(target_plate):
    """Genera todas las matrículas válidas (antiguas y M+) hasta la matrícula objetivo (inclusive)."""
    # Antiguas (primeras letras < 'M')
    for l1 in ALLOWED_FIRST:
        for l2 in ALLOWED_LETTERS:
            for l3 in ALLOWED_LETTERS:
                for num in range(0, 10000):
                    plate = plate_key(num, l1, l2, l3)
                    yield plate, l1
                    if plate == target_plate:
                        return
    # Serie M+
    yield from iter_m_series_until_target(target_plate)

def compute_itv_expiry(l1, serie_m_counter, matriculas_por_dia):
    """Calcula la fecha de caducidad de ITV según si es antigua o M+.
    Devuelve (itv_expiry_iso, is_m_plus_record)."""
    if l1 < 'M':
        if should_be_expired():
            return random_past_date().isoformat(), False
        itv_date = random_future_date()
        if itv_date > MAX_ITV_DATE:
            itv_date = MAX_ITV_DATE
        return itv_date.isoformat(), False
    # M+
    day_index = serie_m_counter // matriculas_por_dia
    fecha_matriculacion = SERIE_M_START + timedelta(days=day_index)
    itv_expiry = fecha_matriculacion.replace(year=fecha_matriculacion.year + 4).isoformat()
    return itv_expiry, True

print("✓ Funciones auxiliares definidas")

## Función Principal de Generación

Genera todas las matrículas hasta la matrícula objetivo:

In [ ]:
def generate_until_target(target_plate=TARGET_PLATE, out_csv=CSV_FILENAME):
    """Genera archivo CSV con matrículas y fechas ITV hasta la matrícula objetivo."""
    total_matriculas_m = count_m_series_until_target(target_plate)
    matriculas_por_dia, total_days = calculate_serie_m_distribution(total_matriculas_m)

    print(f"Serie M+: {total_matriculas_m} matrículas")
    print(f"Distribución: {matriculas_por_dia} matrículas/día durante {total_days} días")
    print(f"\nGenerando archivo {out_csv}...")

    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["plate", "itv_expiry"])
        writer.writeheader()

        serie_m_counter = 0
        total_count = 0

        for plate, l1 in iter_all_plates_until_target(target_plate):
            itv_expiry, is_m = compute_itv_expiry(l1, serie_m_counter, matriculas_por_dia)
            if is_m:
                serie_m_counter += 1

            writer.writerow({"plate": plate, "itv_expiry": itv_expiry})
            total_count += 1

            if total_count % 10000 == 0:
                print(f"  Procesadas: {total_count:,} matrículas...")

    print(f"\n✓ Archivo generado: {out_csv}")
    print(f"  Total matrículas: {total_count:,}")
    return out_csv, True

print("✓ Función generate_until_target refactorizada")

## Ejecutar Generación

Ejecuta la generación de matrículas:

In [ ]:
# Ejecutar generación
print("=" * 60)
print("INICIANDO GENERACIÓN DE MATRÍCULAS")
print("=" * 60)

out_file, finished = generate_until_target(TARGET_PLATE, CSV_FILENAME)

if finished:
    print(f"\n{'='*60}")
    print("✓ GENERACIÓN COMPLETADA EXITOSAMENTE")
    print(f"{'='*60}")
    print(f"Archivo: {out_file}")
    print(f"Matrícula objetivo: {TARGET_PLATE}")
else:
    print(f"\n⚠ No se alcanzó la matrícula objetivo {TARGET_PLATE}")
    print(f"Archivo parcial: {out_file}")

## Verificación de Resultados

Verificamos el archivo generado:

In [ ]:
# Verificar archivo generado
import pandas as pd

# Cargar archivo
df = pd.read_csv(CSV_FILENAME)

print(f"{'='*60}")
print("ESTADÍSTICAS DEL ARCHIVO GENERADO")
print(f"{'='*60}")
print(f"\nTotal de registros: {len(df):,}")
print(f"\nPrimeras 5 matrículas:")
print(df.head())
print(f"\nÚltimas 5 matrículas:")
print(df.tail())

# Estadísticas ITV
df['itv_expiry'] = pd.to_datetime(df['itv_expiry'])
today_pd = pd.Timestamp(TODAY)

caducadas = (df['itv_expiry'] < today_pd).sum()
vigentes = (df['itv_expiry'] >= today_pd).sum()

print(f"\n{'='*60}")
print("DISTRIBUCIÓN ITV")
print(f"{'='*60}")
print(f"ITV caducada:  {caducadas:,} ({caducadas/len(df)*100:.2f}%)")
print(f"ITV vigente:   {vigentes:,} ({vigentes/len(df)*100:.2f}%)")
print(f"\nFecha ITV más lejana: {df['itv_expiry'].max()}")
print(f"Fecha ITV más antigua: {df['itv_expiry'].min()}")